# Dance Performance Analysis System — Gradio Demo
## Interactive Web Interface for Pose Estimation + Genre Classification

**Upload a dance video → get:**
- Skeleton visualization (pose estimation)
- Dance genre prediction (action recognition)
- Confidence scores per genre
- FPS and performance metrics

**Models used:**
- Pose Estimation: YOLO26s-pose (fastest, real-time)
- Genre Classification: ST-GCN (highest accuracy: 98.68%)

**Supported genres:** Break, Pop, Lock, Waack, Mid Hip-hop, LA Hip-hop, House, Krump, Street Jazz, Ballet Jazz

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
!pip install gradio ultralytics opencv-python-headless numpy --quiet
print('All packages installed')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import tempfile
import time
from ultralytics import YOLO
 
yolo_model = YOLO('yolo26s-pose.pt')
print('YOLO26s-pose loaded')
 
GENRES = [
    'Break', 'Pop', 'Lock', 'Waack', 'Mid Hip-hop',
    'LA Hip-hop', 'House', 'Krump', 'Street Jazz', 'Ballet Jazz'
]
 
SKELETON_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]
COLORS_BGR = [
    (255,0,0),(0,255,0),(0,0,255),
    (255,0,255),(0,255,255)
]
 
print('Models ready')


In [ ]:
COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),(5,6),
    (5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]
NUM_JOINTS = 17
 
 
def build_adjacency(num_joints=17):
    A = np.zeros((num_joints, num_joints))
    for i in range(num_joints): A[i,i] = 1
    for i,j in COCO_EDGES: A[i,j] = 1; A[j,i] = 1
    D = np.diag(A.sum(axis=1) ** -0.5)
    return torch.FloatTensor(D @ A @ D)
 
 
class GraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, A):
        super().__init__()
        self.A = A
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        return self.conv(torch.einsum('nctv,vw->nctw', x, self.A.to(x.device)))
 
 
class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.3):
        super().__init__()
        self.gcn = GraphConv(in_ch, out_ch, A)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, (9,1), padding=(4,0), stride=(stride,1)),
            nn.BatchNorm2d(out_ch), nn.Dropout(dropout)
        )
        self.residual = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=(stride,1)),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.residual(x))
 
 
class STGCN(nn.Module):
    def __init__(self, in_channels=2, num_classes=10, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(in_channels * NUM_JOINTS)
        self.blocks  = nn.ModuleList([
            STGCNBlock(in_channels, 64,  A, dropout=dropout),
            STGCNBlock(64,  64,  A, dropout=dropout),
            STGCNBlock(64,  128, A, stride=2, dropout=dropout),
            STGCNBlock(128, 128, A, dropout=dropout),
            STGCNBlock(128, 256, A, stride=2, dropout=dropout),
            STGCNBlock(256, 256, A, dropout=dropout),
        ])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc   = nn.Linear(256, num_classes)
    def forward(self, x):
        N, C, T, V = x.shape
        x = x.permute(0,3,1,2).reshape(N, V*C, T)
        x = self.data_bn(x)
        x = x.reshape(N, V, C, T).permute(0, 2, 3, 1)
        for block in self.blocks: x = block(x)
        return self.fc(self.pool(x).reshape(N, -1))
 
 
STGCN_PATH = '/content/drive/MyDrive/AIST_ActionRec_Final/best_STGCN.pt'
 
stgcn_model = STGCN(in_channels=2, num_classes=10).to(device)
if os.path.exists(STGCN_PATH):
    stgcn_model.load_state_dict(torch.load(STGCN_PATH, map_location=device))
    stgcn_model.eval()
    print(f'ST-GCN loaded from: {STGCN_PATH}')
else:
    print(f'WARNING: {STGCN_PATH} not found — genre classification disabled')


In [ ]:
def normalize_skeleton(keypoints):
    """Normalize by hip center, scale to [-1,1]."""
    xy = keypoints[:, :2].copy()
    if np.all(xy == 0): return np.zeros(34)
    hip_center = (xy[11] + xy[12]) / 2.0
    xy -= hip_center
    max_val = np.abs(xy).max()
    if max_val > 0: xy /= max_val
    return xy.flatten()
 
 
def predict_genre(skeleton_sequence):
    """
    Predict dance genre from skeleton sequence.
    Input: list of (34,) normalized skeletons
    Output: genre name, probabilities dict
    """
    WINDOW = 55
    if len(skeleton_sequence) < WINDOW:
        
        while len(skeleton_sequence) < WINDOW:
            skeleton_sequence.append(np.zeros(34))
 
    
    mid   = len(skeleton_sequence) // 2
    start = max(0, mid - WINDOW // 2)
    window = np.array(skeleton_sequence[start:start+WINDOW])
 
    
    gcn_input = window.reshape(WINDOW, 17, 2).transpose(2, 0, 1)
    gcn_input = torch.FloatTensor(gcn_input).unsqueeze(0).to(device)
 
    with torch.no_grad():
        logits = stgcn_model(gcn_input)
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
 
    pred_idx  = probs.argmax()
    pred_genre = GENRES[pred_idx]
 
    return pred_genre, {g: float(p) for g, p in zip(GENRES, probs)}
 
 
def draw_skeleton_on_frame(frame, keypoints_list):
    """Draw skeleton for all detected persons."""
    img = frame.copy()
    for p_idx, kpts in enumerate(keypoints_list):
        color = COLORS_BGR[p_idx % len(COLORS_BGR)]
        
        for i, j in SKELETON_PAIRS:
            x1, y1, c1 = kpts[i]
            x2, y2, c2 = kpts[j]
            if c1 > 0.3 and c2 > 0.3:
                cv2.line(img, (int(x1),int(y1)), (int(x2),int(y2)), color, 2)
        
        for x, y, c in kpts:
            if c > 0.3:
                cv2.circle(img, (int(x), int(y)), 4, color, -1)
    return img
 
 
def process_video(video_path, max_frames=60):
    """
    Full pipeline: video → pose estimation → genre classification
    Returns: output_video_path, genre, probabilities, stats
    """
    cap = cv2.VideoCapture(video_path)
    fps_vid = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
 
    
    out_path = '/tmp/output_skeleton.mp4'
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'),
                          min(fps_vid, 15), (w, h))
 
    skeleton_sequence = []
    frame_times = []
    n_det_list  = []
    frame_count = 0
    frame_step  = max(1, int(fps_vid // 10))
 
    
    start_frame = int(total * 0.1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
 
    idx = start_frame
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret: break
 
        if (idx - start_frame) % frame_step == 0:
            t0  = time.time()
            res = yolo_model(frame, verbose=False)
            frame_times.append(time.time() - t0)
 
            
            all_kpts = []
            if res[0].keypoints is not None:
                for p_i in range(len(res[0].keypoints)):
                    kpts  = res[0].keypoints[p_i].xy[0].cpu().numpy()
                    confs = res[0].keypoints[p_i].conf[0].cpu().numpy()
                    kpts_conf = np.column_stack([kpts, confs])
                    all_kpts.append(kpts_conf)
 
                
                if all_kpts:
                    best_conf = max(range(len(res[0].boxes)),
                                   key=lambda i: float(res[0].boxes[i].conf[0]))
                    norm = normalize_skeleton(all_kpts[best_conf])
                    skeleton_sequence.append(norm)
 
            n_det_list.append(len(all_kpts))
 
            
            annotated = draw_skeleton_on_frame(frame, all_kpts)
            out.write(annotated)
            frame_count += 1
 
        idx += 1
 
    cap.release()
    out.release()
 
    
    genre = 'Unknown'
    probs = {g: 0.0 for g in GENRES}
    if skeleton_sequence:
        genre, probs = predict_genre(skeleton_sequence)
 
    stats = {
        'fps':         round(1/np.mean(frame_times), 1) if frame_times else 0,
        'avg_det':     round(np.mean(n_det_list), 1) if n_det_list else 0,
        'frames':      frame_count,
        'resolution':  f'{w}×{h}'
    }
 
    return out_path, genre, probs, stats
 
 
print('Processing functions ready')


In [ ]:
import gradio as gr
 
 
def analyze_video(video_path):
    """
    Main Gradio function.
    Input:  video file path
    Output: annotated video, genre label, genre probabilities, stats text
    """
    if video_path is None:
        return None, 'No video uploaded', {}, ''
 
    try:
        out_video, genre, probs, stats = process_video(video_path)
 
        
        stats_text = (
            f'Inference FPS:    {stats["fps"]}\n'
            f'Avg detections:   {stats["avg_det"]} persons/frame\n'
            f'Frames processed: {stats["frames"]}\n'
            f'Resolution:       {stats["resolution"]}\n'
            f'Pose model:       YOLO26s-pose\n'
            f'Genre model:      ST-GCN (98.68% accuracy)'
        )
 
        return out_video, genre, probs, stats_text
 
    except Exception as e:
        return None, f'Error: {str(e)}', {}, ''
 
 
with gr.Blocks(title='Dance Analysis System', theme=gr.themes.Soft()) as demo:
 
    gr.Markdown("""
    # 🕺 Dance Performance Analysis System
    **Upload a dance video** to get automatic pose estimation and genre classification.

    **Supported genres:** Break · Pop · Lock · Waack · Mid Hip-hop · LA Hip-hop · House · Krump · Street Jazz · Ballet Jazz
    """)
 
    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(
                label='Upload Dance Video',
                sources=['upload']
            )
            analyze_btn = gr.Button('Analyze', variant='primary', size='lg')
 
        with gr.Column(scale=1):
            video_output = gr.Video(label='Skeleton Visualization')
 
    with gr.Row():
        with gr.Column(scale=1):
            genre_output = gr.Label(
                label='Predicted Genre',
                num_top_classes=3
            )
        with gr.Column(scale=1):
            stats_output = gr.Textbox(
                label='Performance Metrics',
                lines=7
            )
 
    analyze_btn.click(
        fn=analyze_video,
        inputs=[video_input],
        outputs=[video_output, genre_output, genre_output, stats_output]
    )
 
    gr.Markdown("""
    ---
    **System:** Pose Estimation (YOLO26s-pose) + Genre Classification (ST-GCN)
    **Dataset:** AIST++ — 10 dance genres, 1408 sequences
    **Paper:** Sandra Georgiev, FINKI 2025/2026
    """)
 
demo.launch(share=True, debug=True)


In [ ]:
TEST_VIDEO = '/content/drive/MyDrive/Files (1)/BeatBusterz.mp4'
 
if os.path.exists(TEST_VIDEO):
    print('Testing on BeatBusterz video...')
    out_video, genre, probs, stats = process_video(TEST_VIDEO, max_frames=30)
 
    print(f'Predicted genre: {genre}')
    print(f'\nTop 3 genres:')
    sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]
    for g, p in sorted_probs:
        bar = '█' * int(p * 30)
        print(f'  {g:15s}: {p*100:5.1f}% {bar}')
 
    print(f'\nStats: {stats}')
else:
    print('BeatBusterz video not found')
